# Homework 8 : Neural Networks and Deep Learning

In this homework we gonna build a small PyTorch model that classifies curly and straight hair. The 2026 reference setup freezes the dataset, framework versions, architecture, split, random seed, and evaluation procedure. These settings make the experiment reproducible while still showing why augmentation can change a model's behavior.

In [1]:
!wget -O ../../data/straight-curly-data.zip https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip

--2026-11-23 15:49:08--  https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/405934815/e712cf72-f851-44e0-9c05-e711624af985?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-23T22%3A22%3A51Z&rscd=attachment%3B+filename%3Ddata.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-11-23T21%3A22%3A07Z&ske=2026-11-23T22%3A22%3A51Z&sks=b&skv=2018-11-09&sig=VB6digEjtwEUlxhRl4oWhD9INxX8Esh42KjepoMz6hY%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5NTQ3MjM0OCwibmJmIjoxNzk1NDcwNTQ4LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG

In [2]:
!unzip -q -o ../../data/straight-curly-data.zip -d ../../data/straight-curly-data

The checksum must be:

```text
9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67  data.zip
```

The archive contains a fixed training split and a fixed held-out evaluation
split. It has 1,002 image files. One GIF is kept in the archive for provenance
but `ImageFolder` ignores it, so the training run consumes 1,001 images:

| Split | Curly | Straight | Total |
| --- | ---: | ---: | ---: |
| Train | 410 | 390 | 800 |
| Held-out evaluation | 103 | 98 | 201 |

`torchvision.datasets.ImageFolder` assigns `curly=0` and `straight=1` because
the class directories are sorted alphabetically. Do not create another random
train/test split and do not augment the held-out evaluation images.

## Environment

For the reference run, use Python 3.11 and CPU wheels:

In [3]:
!uv pip install -r requirements-cpu.txt ipykernel

Checked 5 packages in 33ms


The important CPU wheels are `torch==2.9.0+cpu` and
`torchvision==0.24.0+cpu`. Also use
`numpy==2.3.3`, and `Pillow==11.3.0`. The CPU reference defines the grading
baseline. You can experiment on a GPU, but report the CPU run for the numeric
questions.

The directory also contains `reference_train.py`. After extracting the archive,
run `python reference_train.py --data-dir data` to produce both history files
with the complete reference procedure.

## Reproducibility rules

Run this before constructing datasets, data loaders, the model, or the
optimizer:

```python
import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")
```

Use `num_workers=0` for both loaders. Give the training loader its own fixed
generator:

```python
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)
```

These settings remove the worker-order and GPU/cuDNN differences that made the
2025 scalar answers move between runs.

## Model and preprocessing

Use this transform for the baseline training set and, separately, for the
held-out evaluation set:

```python
from torchvision import transforms

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])
```

Create the model with exactly this structure:

```python
import torch.nn as nn


class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)
```

The model returns logits. Use `nn.BCEWithLogitsLoss()` and apply
`torch.sigmoid` only when calculating predictions or serving the model. Do not
put a sigmoid module in `HairModel` as well; applying it twice was an
ambiguity in the previous assignment.

Use this optimizer and loss:


```python
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)
```

---

## Code

In [4]:
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import torch.nn as nn

In [5]:
import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [6]:
evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [7]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

#### Simple Preprocessing

In [8]:
input_size = 224

# ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Simple transforms - just resize and normalize
train_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

#### Create DataLoads

In [9]:
train_dataset = datasets.ImageFolder("../../data/straight-curly-data/data/train", transform=evaluation_transform)
evaluation_dataset = datasets.ImageFolder("../../data/straight-curly-data/data/test", transform=evaluation_transform)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
eval_loader = DataLoader(evaluation_dataset, batch_size=32, shuffle=False)

In [10]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

#### Train model

In [11]:
model = HairModel()
model.to(device)

HairModel(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (hidden): Linear(in_features=313632, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
)

In [12]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

## Question 1 — loss and output contract

Which loss matches the model above?

Respone : `BCEWithLogitsLoss()`

## Question 2 — parameter count

What is the total number of trainable parameters? Count biases as well and use
`sum(p.numel() for p in model.parameters())`.

In [13]:
total_params = sum(p.numel() for p in model.parameters())
bias_params = sum(p.numel() for name, p in model.named_parameters() if "bias" in name)
print(f"Total parameters: {total_params:,} | Bias parameters: {bias_params:,}")

Total parameters: 20,073,473 | Bias parameters: 97


## Training metrics
<details>
<br>
Train for 10 epochs with the baseline transform. For every epoch, evaluate the
held-out split with `model.eval()` and `torch.no_grad()`. Store these four
series in a JSON file named `history_baseline.json`:

```python
{
    "train_loss": [...],
    "train_accuracy": [...],
    "evaluation_loss": [...],
    "evaluation_accuracy": [...],
}
```

For accuracy, use `(torch.sigmoid(logits) >= 0.5)`. Use the number of examples,
not the number of batches, when averaging loss and accuracy.

In [14]:
import json

def train_and_evaluate(model, train_loader, evaluation_loader, criterion, optimizer, epochs=10):
    history = {"train_loss": [], "train_accuracy": [], "evaluation_loss": [], "evaluation_accuracy": []}

    for epoch in range(epochs):
        # train
        model.train()
        total_loss, correct, n = 0.0, 0, 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)   # shape (batch, 1)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)        # weight by examples
            correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()
            n += images.size(0)
        history["train_loss"].append(total_loss / n)
        history["train_accuracy"].append(correct / n)

        # evaluate
        model.eval()
        total_loss, correct, n = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in evaluation_loader:
                images = images.to(device)
                labels = labels.float().unsqueeze(1).to(device)
                logits = model(images)
                total_loss += criterion(logits, labels).item() * images.size(0)
                correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()
                n += images.size(0)
        history["evaluation_loss"].append(total_loss / n)
        history["evaluation_accuracy"].append(correct / n)

        print(f"Epoch {epoch+1}: train_acc={history['train_accuracy'][-1]:.4f} "
              f"eval_acc={history['evaluation_accuracy'][-1]:.4f}")

    return history


## Question 3 — baseline accuracy

Calculate the median of `train_accuracy` in `history_baseline.json`. Submit the
value rounded to two decimal places. The grader accepts an absolute error of
`0.05`; the answer is not selected from a fixed option list.

In [15]:
histories = {}
histories["baseline"] = train_and_evaluate(model, train_loader, evaluation_loader, criterion, optimizer, epochs=10)

with open("history_baseline.json", "w") as f:
    json.dump(histories["baseline"], f, indent=2)

Epoch 1: train_acc=0.6450 eval_acc=0.6567
Epoch 2: train_acc=0.7175 eval_acc=0.6517
Epoch 3: train_acc=0.7550 eval_acc=0.6219
Epoch 4: train_acc=0.7913 eval_acc=0.6915
Epoch 5: train_acc=0.8087 eval_acc=0.6617
Epoch 6: train_acc=0.8075 eval_acc=0.7114
Epoch 7: train_acc=0.8363 eval_acc=0.6716
Epoch 8: train_acc=0.8850 eval_acc=0.7214
Epoch 9: train_acc=0.9100 eval_acc=0.7164
Epoch 10: train_acc=0.9387 eval_acc=0.6368


In [18]:
with open("history_baseline.json") as f:
    history = json.load(f)

print("Median - Train_Accuracy : ", round(np.median(history["train_accuracy"]), 2))


Median - Train_Accuracy :  0.81


## Question 4 — baseline loss

Calculate the population standard deviation of `train_loss` in
`history_baseline.json` (for NumPy, use `np.std(values, ddof=0)`). Submit the
value rounded to three decimal places. The grader accepts an absolute error of
`0.02`.

In [19]:
with open("history_baseline.json") as f:
    history = json.load(f)

print("Standard Deviation - Train_Loss : ", round(np.std(history["train_loss"], ddof=0), 3))

Standard Deviation - Train_Loss :  0.139


## Data augmentation

Keep the same model and optimizer. Replace only the training transform with:

In [20]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

Do not recreate the model or optimizer. Train for 10 additional epochs and
write only those ten epochs to `history_augmented.json`. Continue using the
unchanged, deterministic `evaluation_dataset` and `evaluation_loader` for
every evaluation.

#### New Augmented Train

In [21]:
train_dataset_augmented = datasets.ImageFolder(
    "../../data/straight-curly-data/data/train",
    transform=train_transform_augmented,
)

train_loader_augmented = DataLoader(
    train_dataset_augmented,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)

## Question 5 — augmented evaluation loss

Calculate the mean of `evaluation_loss` in `history_augmented.json`. Submit the
value rounded to three decimal places. The grader accepts an absolute error of
`0.02`.

In [22]:
histories["augmented"] = train_and_evaluate(
    model, train_loader_augmented, evaluation_loader, criterion, optimizer, epochs=10
)

with open("history_augmented.json", "w") as f:
    json.dump(histories["augmented"], f, indent=2)

Epoch 1: train_acc=0.6400 eval_acc=0.6866
Epoch 2: train_acc=0.6787 eval_acc=0.6766
Epoch 3: train_acc=0.6613 eval_acc=0.7114
Epoch 4: train_acc=0.7150 eval_acc=0.6816
Epoch 5: train_acc=0.7113 eval_acc=0.7065
Epoch 6: train_acc=0.7275 eval_acc=0.7065
Epoch 7: train_acc=0.7037 eval_acc=0.7114
Epoch 8: train_acc=0.7250 eval_acc=0.6965
Epoch 9: train_acc=0.7412 eval_acc=0.6517
Epoch 10: train_acc=0.7688 eval_acc=0.7463


In [23]:
with open("history_augmented.json") as f:
    history_aug = json.load(f)

print("Mean - Evaluation_Loss : ", round(np.mean(history_aug["evaluation_loss"]), 3))

Mean - Evaluation_Loss :  0.611


## Question 6 — augmented evaluation accuracy

Calculate the mean of the last five values in `evaluation_accuracy` in
`history_augmented.json` (epochs 6 through 10 of the augmented phase). Submit
the value rounded to two decimal places. The grader accepts an absolute error
of `0.05`.

The tolerances cover minor CPU wheel differences, while the fixed seed, loader,
split, architecture, and inference transform prevent the large uncontrolled
variation from the 2025 exercise.

In [24]:
print("Mean - Last 5 Evaluation_Accuracy : ", round(np.mean(history_aug["evaluation_accuracy"][-5:]), 2))

Mean - Last 5 Evaluation_Accuracy :  0.7
